# Formative 3 — Probability Distributions, Bayes & Gradient Descent
**Group:** _Formative 3 Group 25_  <br>
**Group Members:**  <br>
 _Teta Nelly_ ,  <br>
 _Elie Kuradusenge_ ,  <br>
_Fabrice Mbarushimana_

All logic lives in `src/` (modular, DRY); this notebook only calls it and shows results.

In [1]:
import os

# 1. Setup repository configuration
REPO_URL = "https://github.com"
REPO_NAME = "Probability_Distributions_Cohort3_Team25"

# 2. Check and clone if it doesn't exist
if not os.path.exists(f"/content/{REPO_NAME}"):
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}
else:
    print(f"Repository '{REPO_NAME}' is already available in this session.")

# 3. Change directory to the repository root
%cd /content/{REPO_NAME}
import sys, numpy as np, pandas as pd
sys.path.insert(0, "src")
import em_gmm as em, bayes, gradient_descent as gd, plotting as viz
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

## Part 1 — Mixture of two Gaussians with EM (from scratch)
**Data:** Galton heights. Unlabelled pool = every child height (934) + one mother height per family (205, de-duplicated so a
mother is not counted once per child). We *hide* the labels and let EM find two Gaussians.

In [3]:
g = pd.read_csv("data/GaltonFamilies.csv")
children = g.childHeight.values
mothers  = g.drop_duplicates("family").mother.values
x = np.concatenate([children, mothers])
true_is_child = np.r_[np.ones(len(children)), np.zeros(len(mothers))]   # kept ONLY for after-the-fact evaluation
print(len(children), "children +", len(mothers), "mothers =", len(x), "heights;  global mean =", round(x.mean(), 3))

FileNotFoundError: [Errno 2] No such file or directory: 'data/GaltonFamilies.csv'

### Why not just split at the global mean?
A hard cut at the global mean assumes the two groups never overlap and that each side is "pure". Mothers and children overlap
heavily, so the cut (1) **misclassifies** every point on the wrong side, (2) **biases the means**: each pile's mean is pulled away
from the true group mean because the tails of the other group are inside it, (3) ignores the **different spreads and group sizes**
(it forces equal-ish variances and weights), and (4) gives no uncertainty — a height of 66.2 is treated exactly like 60 or 75.
EM instead gives **soft assignments** (responsibilities) and estimates μ, σ², π jointly by maximising the likelihood.

In [ ]:
# initialisation (no labels needed): means one SD either side of the global mean, equal variance, equal weights
mu0, var0, pi0 = [x.mean() - x.std(), x.mean() + x.std()], [x.var()] * 2, [0.5, 0.5]
mu, var, pi, hist = em.fit_em(x, mu0, var0, pi0, tol=1e-6)
print("converged after", len(hist) - 1, "iterations")

# tracking table (initial state + first two iterations, plus the final state)
rows = [h for h in hist[:3]] + [hist[-1]]
table = pd.DataFrame([{"Iteration": ("0 (init)" if h["iter"] == 0 else h["iter"]) if h is not hist[-1] or h["iter"] < 3 else f'{h["iter"]} (final)',
                       "μ1": h["mu"][0], "μ2": h["mu"][1], "σ1²": h["var"][0], "σ2²": h["var"][1],
                       "π1": h["pi"][0], "π2": h["pi"][1], "Log-Likelihood": h["ll"]} for h in rows])
table

**Reading the table.** Component 1 = the *shorter* Gaussian, Component 2 = the *taller* one. EM was never told who is a mother
or a child, so it names clusters by height, not by role (see the check below).

In [ ]:
viz.plot_mixture(x, mu, var, pi, ["Component 1 (shorter)", "Component 2 (taller)"],
                 "Two-Gaussian mixture found by EM", path="figures/em_fit.png")
viz.plot_lines([h["iter"] for h in hist[:40]], [[h["ll"] for h in hist[:40]]], ["log-likelihood"],
               "EM: log-likelihood rises monotonically (first 40 iterations)", "log-likelihood", path="figures/em_ll.png")

### Classification demo — exact posteriors for a test height

In [ ]:
def classify(h):
    p = em.posterior(h, mu, var, pi)
    print(f"Height {h} in:  P(Component 1, shorter) = {p[0]:.4f}   P(Component 2, taller) = {p[1]:.4f}   -> sum = {p.sum():.1f}")
    return p
TEST_HEIGHT = 66.0          # <- coach's height goes here
classify(TEST_HEIGHT)
for h in (60, 64, 68, 72): classify(h)

### Honest check against the hidden labels
Mean-split vs EM hard assignment vs the true mother/child labels. Because **children include sons (~69 in) and daughters
(~64 in)**, the two Gaussians EM finds are "shorter people" vs "taller people", which is not exactly "mothers vs children".

In [ ]:
resp = em.e_step(x, mu, var, pi)
em_comp = resp.argmax(axis=1)                                  # 0 = shorter, 1 = taller
split_comp = (x > x.mean()).astype(int)
print("Mean-split group means :", x[split_comp == 0].mean().round(3), x[split_comp == 1].mean().round(3))
print("EM component means     :", mu.round(3))
print("\nCross-tab of EM component vs true label:")
print(pd.crosstab(pd.Series(em_comp, name="EM component (0=short,1=tall)"),
                  pd.Series(np.where(true_is_child == 1, "child", "mother"), name="true label")))
print("\nSons / daughters / mothers by EM component:")
who = np.r_[g.gender.values, ["mother"] * len(mothers)]
print(pd.crosstab(pd.Series(em_comp, name="component"), pd.Series(who, name="who")))
soft = resp[:, 1]; print("\nPoints with 0.2 < P(tall) < 0.8 (genuinely ambiguous):", int(((soft > .2) & (soft < .8)).sum()), "of", len(x))

## Part 2 — Bayes' theorem on IMDb reviews (standard library only)
**Choice:** we compute **P(Positive | keyword)** only. A review "contains" a keyword if the word appears at least once
(document presence; HTML `<br />` stripped, lower-cased, word-tokenised with `re`).
Positive keywords: *excellent, wonderful, amazing, perfect* — strong, unambiguous praise words.
Negative keywords: *worst, awful, boring, waste* — strong criticism words. (Negatives should give a **low** P(Positive | kw).)

In [ ]:
reviews = bayes.load_reviews("data/IMDB_Dataset.csv")      # IMDB_Dataset.csv: put the Kaggle file in data/
POS = ["excellent", "wonderful", "amazing", "perfect"]; NEG = ["worst", "awful", "boring", "waste"]
tab = pd.DataFrame(bayes.keyword_table(reviews, POS + NEG)).set_index("keyword")
tab.columns = ["Prior P(Pos)", "Likelihood P(kw|Pos)", "Marginal P(kw)", "Posterior P(Pos|kw)", "Direct count check", "# reviews with kw"]
tab

In [ ]:
viz.plot_bars(tab.index, tab["Posterior P(Pos|kw)"], "P(Positive | keyword)", "posterior",
              ["seagreen"] * 4 + ["indianred"] * 4, path="figures/bayes.png")
# sanity: Bayes result equals the direct frequency
assert np.allclose(tab["Posterior P(Pos|kw)"], tab["Direct count check"])

**Interpretation.** With a 50/50 prior, "excellent" raises P(Positive) to ≈0.81 and "waste" drops it to ≈0.07. The likelihoods
are small (a given word appears in few reviews) but the *ratio* likelihood/marginal is what moves the prior.

## Part 3 — Manual gradient descent (see PDF) and Part 4 — in code
Same problem: ŷ = X m + b·1, X = [[1,3],[4,10]], y = [5,6], m₀ = [-1,2], b₀ = 1, α = 0.01.
The hand calculations are in **`part3_gradient_descent_calculations.pdf`**; here they are reproduced step by step.

In [ ]:
X = np.array([[1, 3], [4, 10]], float); y = np.array([5, 6], float)
m0, b0, LR = np.array([-1.0, 2.0]), 1.0, 0.01
y_hat = gd.predict(X, m0, b0); err = y_hat - y
print("ŷ  =", y_hat, "\nerror =", err, "\nJ =", gd.mse(X, y, m0, b0))
gm, gb = gd.analytic_gradient(X, y, m0, b0)
print("analytic  ∇m =", gm, " ∂b =", gb)
# SciPy derivative of the cost function as an independent check
cost = lambda p: gd.mse(X, y, p[:-1], p[-1])
print("SciPy     [∂m1, ∂m2, ∂b] =", gd.scipy_gradient(cost, [*m0, b0]).round(6))

In [ ]:
N_UPDATES = 3                                                       # = number of group members
m_fin, b_fin, h5 = gd.gradient_descent(X, y, m0, b0, LR, N_UPDATES)
pd.DataFrame([{"update": r["k"], "m1": r["m"][0], "m2": r["m"][1], "b": r["b"], "J": r["cost"],
               "ŷ": np.round(r["y_hat"], 4)} for r in h5])

In [ ]:
# longer run for the plots
M, B, H = gd.gradient_descent(X, y, m0, b0, LR, 300)
ks = [r["k"] for r in H]
viz.plot_lines(ks, [[r["m"][0] for r in H], [r["m"][1] for r in H], [r["b"] for r in H]], ["m1", "m2", "b"],
               "Parameters over iterations", "value", path="figures/params.png")
viz.plot_lines(ks, [[r["cost"] for r in H]], ["MSE J"], "Error over iterations", "MSE", path="figures/error.png", logy=True)
print("final m =", M.round(4), " b =", round(B, 4), " J =", round(H[-1]["cost"], 4))
print("final predictions ŷ =", gd.predict(X, M, B).round(4), "(targets", y, ")")

In [ ]:
# cross-check against the closed-form least-squares solution
A = np.c_[X, np.ones(2)]
print("lstsq (min-norm) :", np.linalg.lstsq(A, y, rcond=None)[0].round(4))
print("GD after 300 it. :", np.r_[M, B].round(4), "-> still converging along a flat valley (X has only 2 points, 3 parameters)")

**Trend.** J falls every update (61 → 6.8 → 2.7 → 2.4 → 2.3 → …). m₂ drops fast then wobbles slightly because of the
correlated features; b creeps up. With 2 samples and 3 parameters there are infinitely many exact solutions, so GD slides along a
shallow valley toward J → 0 rather than a single point.